# 01d GAMSPy

[GAMSPy](https://gamspy.readthedocs.io) is the Python interface to GAMS. It has no PIPS-IPM++
solver of its own, so this notebook uses a small bridge module, `pipsipmpp_interface.py`, which
sits next to it. The bridge lets GAMS generate the model, reads it through the GAMS modelling
object (GMO), and can then write it as parquet or hand it to PIPS-IPM++. After a solve it writes
the answer back onto the GAMSPy symbols, so the model reads just as after an ordinary GAMS solve.

The block of a variable is its GAMS `.stage` attribute, which GAMS users may know from stochastic
programming and from the PIPS-IPM++ driver for gdx files.

| sections | run on |
|---|---|
| 1 to 4: build, export, visualize, change the model, reference solve with GAMS | every machine |
| 5: solve with PIPS-IPM++ | Linux or WSL with the `solver` extra |
| 6: more than one process with MPI | Linux or WSL with the `solver` extra |

In [ ]:
import workshop

workshop.check()

## 1. The model

The sample model from [00](00-start-here.ipynb). The data (costs, profiles, the emission budget)
lives in `sample_data.py` next to this notebook, so every framework computes the same numbers.

The cell below writes the model into a file, `gamspy_model.py`, instead of running it. That way
this notebook and the parallel run in section 6 use the very same code. The cell after it loads
the file.

The annotation is in the middle of `energy_model`:

- `cap.stage[g] = 1`: stage 1 is the root,
- `gen.stage[g, t] = ...`: stage `k + 1` is leaf `k`, so four leaves are the stages 2 to 5.

In [ ]:
%%writefile gamspy_model.py
from gamspy import Container, Equation, Model, Parameter, Sense, Set, Sum, Variable

from sample_data import STORAGE, TECHS, budget, profiles

ROOT_STAGE = 1


def energy_model(T=24, n_blocks=4, techs=TECHS, storage=False):
    demand, avail = profiles(T)
    hours = -(-T // n_blocks)                                     # hours per block, rounded up

    m = Container()
    t = Set(m, "t", records=[str(i) for i in range(T)])
    g = Set(m, "g", records=list(techs))
    p_dem = Parameter(m, "demand", domain=t, records=[[str(i), demand[i]] for i in range(T)])
    p_av = Parameter(m, "avail", domain=[g, t],
                     records=[[name, str(i), avail[techs[name][3]][i]]
                              for name in techs for i in range(T)])
    p_inv, p_var, p_emi = (
        Parameter(m, label, domain=g, records=[[name, techs[name][k]] for name in techs])
        for k, label in enumerate(["invcost", "varcost", "emission"]))

    cap = Variable(m, "cap", domain=g, type="positive")
    gen = Variable(m, "gen", domain=[g, t], type="positive")
    supply = Sum(g, gen[g, t])
    cost = Sum(g, p_inv[g] * cap[g]) + Sum((g, t), p_var[g] * gen[g, t])

    # the annotation: stage 1 is the root, stage k + 1 is leaf k
    p_stage = Parameter(m, "stage", domain=t,
                        records=[[str(i), ROOT_STAGE + 1 + i // hours] for i in range(T)])
    cap.stage[g] = ROOT_STAGE
    gen.stage[g, t] = p_stage[t]

    if storage:
        sto_cap = Variable(m, "sto_cap", type="positive")          # stage 1: root
        charge = Variable(m, "charge", domain=t, type="positive")
        discharge = Variable(m, "discharge", domain=t, type="positive")
        soc = Variable(m, "soc", domain=t, type="positive")
        for v in (charge, discharge, soc):
            v.stage[t] = p_stage[t]
        # the level carries over from the previous hour; soc[t.lag(1)] is 0 in the first
        soc_level = Equation(m, "soc_level", domain=t)
        soc_level[t] = soc[t] == soc[t.lag(1)] + STORAGE["efficiency"] * charge[t] - discharge[t]
        soc_limit = Equation(m, "soc_limit", domain=t)
        soc_limit[t] = soc[t] <= sto_cap
        supply = supply + discharge[t] - charge[t]
        cost = cost + STORAGE["investment"] * sto_cap

    balance = Equation(m, "balance", domain=t)
    balance[t] = supply == p_dem[t]
    limit = Equation(m, "limit", domain=[g, t])
    limit[g, t] = gen[g, t] <= p_av[g, t] * cap[g]
    carbon = Equation(m, "carbon")                                   # linking
    carbon[...] = Sum((g, t), p_emi[g] * gen[g, t]) <= budget(T)

    return Model(m, "expansion", equations=m.getEquations(),
                 problem="LP", sense=Sense.MIN, objective=cost)

In [ ]:
%run gamspy_model.py

model = energy_model()

# GAMS keeps a variable's .stage in the same field as its .scale, so it appears as "scale"
gen = model.container["gen"].records.rename(columns={"scale": "stage"})
print(gen.assign(hour=gen["t"].astype(int)).groupby("stage")["hour"].agg(["min", "max", "count"]))
print()
print("cap records:", model.container["cap"].records)

Each leaf stage (2 to 5) holds six hours of both technologies. And `cap` has no records at all:
GAMS only stores a record when something differs from its default, and stage 1 is the default.
A variable you never annotate therefore lands in the root.

<details><summary>💡 Hint: I changed the model. Why did nothing happen?</summary>

`%%writefile` saves the code and `%run` loads it. After editing the first cell, run both again,
then `energy_model()` builds the new model.
</details>

<details><summary>💡 Hint: there is no <code>stage</code> column, only <code>scale</code>?</summary>

A long-standing GAMS detail: a variable's `.stage` and `.scale` share one field, so the stage you
set shows up in the records as `scale`. It does no harm as long as GAMS scaling stays off, which is
the default (`scaleOpt = 0`). If you do scale a model, you cannot also use stages this way.
</details>

<details><summary>💡 Hint: why does the root start at stage 1 and not 0?</summary>

GAMS numbers stages from 1, and a variable you never set keeps the default of 1, the root. That is
the safe place for anything you are unsure about. The bridge subtracts one, so stage `k + 1`
becomes PIPS-IPM++ block `k`.
</details>

## 2. Export it and look at the matrix

`pips.write_parquet` lets GAMS generate the model and writes it the way PIPS-IPM++ reads it: the
matrix, the bounds and costs, and the block of every row and column. It does not need the solver
or MPI, so it runs on every machine.

- `layout="monolithic"` writes one set of files for the whole model. (`"distributed"` writes one
  set per block, see [04](04-deep-dive.ipynb).)
- `names=True` keeps the GAMS names, `gen(wind,17)`, for every row and column.
- The number of leaves is taken from the highest stage.

In [ ]:
from pathlib import Path
import pipsipmpp_interface as pips

Path("data").mkdir(exist_ok=True)


def export(model, name, **annotation):
    return pips.write_parquet(model, f"data/{name}", layout="monolithic", names=True,
                              **annotation)


stem = export(energy_model(), "gamspy_base")
print(sorted(p.name for p in Path("data").glob("gamspy_base.*")))

`workshop.visualize` runs `pipstools visualize` on the files, shows the picture, and prints the
numbers that decide whether a structure is good:

In [ ]:
workshop.visualize(stem)

How to read it:

- Rows and columns are sorted by block. Each dot is a nonzero.
- The narrow band on the left holds the 2 root columns, `cap(wind)` and `cap(gas)`. Its dots are
  the capacity limits of every hour, which use the capacity of their technology.
- The four coloured squares are the leaf blocks, six hours each: 12 `gen` columns and 18 rows
  (6 balance, 12 limit).
- The grey row at the bottom is the one linking row, the carbon budget. It touches every leaf.

This is the structure from the table in 00, now drawn from the actual matrix.

<details><summary>💡 Hint: where is the objective variable?</summary>

GAMS models carry an objective variable and an equation that defines it. The bridge removes both
and hands PIPS-IPM++ the cost vector instead, so the files hold exactly the model from 00. If you
export the model with GAMS itself, for example as a Jacobian GDX, the objective equation is still
there and shows up as a second linking row.
</details>

## 3. Change the model and watch the matrix

Each exercise changes the model once. Before you run a cell, predict what happens to the three
numbers: root columns, columns per leaf, linking rows. Then compare with the picture.

### Exercise 1: more hours

Two days instead of one, still in 4 blocks, and then in 8.

In [ ]:
workshop.visualize(export(energy_model(T=48), "gamspy_T48"))

In [ ]:
workshop.visualize(export(energy_model(T=48, n_blocks=8), "gamspy_T48_8b"))

The root stays at 2 columns however many hours you add, and there is still one linking row. More
hours make each leaf larger (with 4 blocks) or give more leaves (with 8). This is the property
that makes energy models a good fit: they grow along time, and time goes into the leaves.

Try `T=168`, a week, with 7 blocks of one day each.

### Exercise 2: another technology

`sample_data.py` describes a technology as one row: investment cost, variable cost, emissions and
availability profile. `SOLAR` is a row prepared for you; adding it is one line.

In [ ]:
from sample_data import SOLAR, TECHS

with_solar = TECHS | {"solar": SOLAR}
print(with_solar)
workshop.visualize(export(energy_model(techs=with_solar), "gamspy_solar"))

One more root column, `cap(solar)`, and every leaf grows by one `gen` column and one limit row
per hour. The linking row stays alone: the carbon budget now touches solar too, but it was
linking already.

Invent your own technology: a tuple `(investment, variable, emission, profile)` with one of the
profiles `"wind"`, `"solar"` or `"firm"`. A cheap, high emission technology such as `(0.2, 0.8, 0.9, "firm")`
changes the answer a lot and the structure not at all.

### Exercise 3: storage

`energy_model(storage=True)` adds the storage from 00. Read the `if storage:` part of the model
first. The equation to look at is the level:

```python
soc_level[t] = soc[t] == soc[t.lag(1)] + efficiency * charge[t] - discharge[t]
```

Inside a block this row is local. In the first hour of a block `soc[t.lag(1)]` belongs to the
previous block. Predict the number of linking rows, then run:

In [ ]:
workshop.visualize(export(energy_model(storage=True), "gamspy_storage"))

Four linking rows: the carbon budget, plus one storage level per block boundary. Those three are
2-link rows: each touches only two neighbouring blocks, and you can see them in the picture as
short strokes at the bottom, right at the corners between two blocks. PIPS-IPM++ handles them
much more cheaply than the carbon budget, which touches every block.

The storage capacity `sto_cap` is the third root column. Its stage was never set, so it kept the
default, 1.

### Exercise 4: restage the model

Stages are ordinary GAMS attributes, so you can change them after the model is built. Here every
fourth hour goes to the same block. A perfectly valid split, and a very poor one for a model with
storage:

In [ ]:
from gamspy import Parameter

model = energy_model(storage=True)
c = model.container
t, g = c["t"], c["g"]

round_robin = Parameter(c, "round_robin", domain=t,
                        records=[[str(i), 2 + i % 4] for i in range(24)])
c["gen"].stage[g, t] = round_robin[t]
for name in ("charge", "discharge", "soc"):
    c[name].stage[t] = round_robin[t]

workshop.visualize(export(model, "gamspy_roundrobin"))

Now every storage level links two blocks, and the bottom border is as tall as a leaf. Most of
those rows are still 2-link (block 1 to 2, 2 to 3, 3 to 4). The ones from block 4 back to block 1
are not, because blocks 4 and 1 are not neighbours.

The model is the same and its answer is the same. Only the split changed, and with it how much
work the borders cost.

### Exercise 5: a model with no stages at all

Not every GAMS model sets `.stage`. The bridge can then derive the blocks from the names GAMS
gives the columns: `gen(wind,17)`, `soc(17)`. A regular expression captures the hour, and every
distinct captured value is put into blocks in order. Columns without an hour, `cap(wind)` and
`sto_cap`, match nothing and stay in the root.

This is the regex method of pipstools, which [03](03-structure-detection.ipynb) uses on a much
larger model. It runs everywhere.

In [ ]:
import re

pattern = r"[(,](\d+)\)$"             # digits after "(" or ",", right before the closing ")"
for name in ("gen(wind,17)", "soc(17)", "cap(wind)", "sto_cap"):
    found = re.search(pattern, name)
    print(f"{name:14s} -> {found.group(1) if found else 'no match, root'}")

model = energy_model(storage=True)
workshop.visualize(export(model, "gamspy_regex", n_blocks=4, annotation="regex", regex=pattern))

The same structure as the stages gave in exercise 3, derived from the names alone. The two lines
`Block 1: 0, 1, 2, ... 2, 3, 4` above the summary come from pipstools and show which captured
values went into the first and the last block.

### Exercise 6: your own change

Some ideas, each with a prediction to check against the picture:

- Make the storage cyclic, so the last hour feeds the first: replace `soc[t.lag(1)]` by
  `soc[t.lag(1, "circular")]`, run both cells of section 1 again, and export
  `energy_model(storage=True)`.
- Blocks of different lengths: change the `p_stage` records.
- Try `annotation="hypergraph"` instead of the regex. It partitions the matrix without looking at
  names (Linux and macOS, there is no Windows build of the partitioner).

<details><summary>💡 Answer: what does the cyclic storage do?</summary>

It adds one linking row that ties the last hour of block 4 to the first hour of block 1. That row
touches two blocks, but not two neighbouring ones, so it is not 2-link: the summary reports 5
linking rows, of which 3 are 2-link. One such row is harmless. A model with many of them is worth
reformulating.
</details>

## 4. The answer, with GAMS

Before solving with PIPS-IPM++, get a reference answer from the LP solver that comes with GAMSPy,
which runs everywhere:

In [ ]:
reference = {}
for label, model in (("base", energy_model()), ("storage", energy_model(storage=True))):
    model.solve()
    reference[label] = model.objective_value
    cap = model.container["cap"].records
    print(f"{label:8s} objective {model.objective_value:.6f}   capacities",
          dict(zip(cap["g"], cap["level"].round(4))))

Storage lowers the cost from about 2.74 to about 2.12: it shifts wind into the hours without
wind, and no gas is built at all.

## 5. Solve with PIPS-IPM++

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped.

`pips.solve` takes a function that builds the model rather than the model itself. It calls it on
process 0 only, so GAMS generates the model once however many processes the job has.

In [ ]:
%%requires solver
model = energy_model()
solution = pips.solve(lambda: model, options={"SILENT": True})

In [ ]:
%%requires solver
cap = model.container["cap"].records
print(solution.status.name, "| optimal:", solution.optimal)
print(f"objective   {model.objective_value:.6f}   (GAMS {reference['base']:.6f})")
print("capacities ", dict(zip(cap["g"], cap["level"].round(4))))
print("carbon dual", round(float(model.container["carbon"].records["marginal"].iloc[0]), 4))

The same answer as the GAMS reference, to about six figures. PIPS-IPM++ is an interior-point
method and stops within a tolerance of the optimum, so the last digits may differ.

The levels and marginals are on the GAMSPy symbols, as after any GAMS solve: `cap.records`,
`cap.l[...]`, `.m` on equations. `solve` also returns them as an object, for when reading them
off the container is not what you want:

In [ ]:
%%requires solver
print(solution.iterations, "iterations")
print(dict(list(solution.levels().items())[:3]))      # values by GAMS name

The solve cell shows what PIPS-IPM++ prints: its parameters, a presolve summary and the residuals
at the end. `SILENT` removes most of the iteration log but not those, which is why the results
get a cell of their own.

### The split changes the work, never the answer

The storage model with a few block counts, and with the round robin and the regex structure from
exercises 4 and 5:

In [ ]:
%%requires solver
splits = {}
for n_blocks in (2, 4, 6, 12):
    model = energy_model(storage=True, n_blocks=n_blocks)
    pips.solve(lambda: model, options={"SILENT": True})
    splits[f"{n_blocks} contiguous blocks"] = model.objective_value

model = energy_model(storage=True)
pips.solve(lambda: model, n_blocks=4, annotation="regex", regex=pattern,
           options={"SILENT": True})
splits["regex, 4 blocks"] = model.objective_value

In [ ]:
%%requires solver
for label, objective in splits.items():
    print(f"{label:20s} objective {objective:.6f}")
print(f"{'GAMS':20s} objective {reference['storage']:.6f}")

If changing the split ever changes your objective beyond the solver tolerance, the annotation has
a bug.

<details><summary>💡 Hint: <code>n_blocks</code> is not passed to <code>solve</code>. Where does it come from?</summary>

From the stages. The bridge takes the highest stage it finds, so stages 1 to 5 mean four leaves.
Pass `n_blocks=` to insist on a number, for instance when one leaf came out empty, or together
with `annotation=` when the structure is derived.
</details>

## 6. More than one process

A notebook is a single process, so everything above solved on one. To use the decomposition, the
same code runs under MPI as a script. Only process 0 builds and generates the model; the others
wait for the blocks they own. The call is collective, so every process has to reach `pips.solve`.

In [ ]:
%%writefile gamspy_parallel.py
from mpi4py import MPI
import pipsipmpp_interface as pips
from gamspy_model import energy_model

comm = MPI.COMM_WORLD
model = energy_model(storage=True) if comm.Get_rank() == 0 else None   # GAMS generates once
solution = pips.solve(lambda: model, comm=comm, options={"SILENT": True})
if comm.Get_rank() == 0:
    print(f"ranks={comm.Get_size()} {solution.status.name} "
          f"objective={model.objective_value:.6f}")

In [ ]:
%%requires solver
for ranks in (1, 2, 4):
    workshop.mpirun("gamspy_parallel.py", ranks, show="ranks=")

The same objective on one, two and four processes, within the solver tolerance. The work is
split differently, so the last digits can differ, and so can the number of iterations.

`workshop.mpirun` runs `mpiexec -n 4 python gamspy_parallel.py` for you; in a terminal you can type that yourself.

<details><summary>💡 Hint: how many processes should I use?</summary>

At most one per leaf block, since PIPS-IPM++ refuses more processes than leaves. Fewer is fine;
leaves then share a process. On a laptop, the number of CPU cores is the useful upper limit.
</details>

## 7. Now your own model

If your GAMS model is already annotated for the PIPS-IPM++ link of GAMS, its variable stages work
here as they are. Otherwise:

1. Decide what the blocks are: time slices, scenarios, regions.
2. Set `.stage` on the variables, 1 for everything shared and `k + 1` for leaf `k`, or derive the
   blocks from the names with `annotation="regex"` as in exercise 5.
3. Export and visualize the model as in section 2 before solving anything big.

<details><summary>💡 Hint: GAMSPy refuses my larger model with a licence message</summary>

Out of the box GAMSPy runs on the GAMS demo licence, which limits the model size
(`gamspy show license` tells you which licence you have). Academics can get a free GAMSPy licence
and install it with `gamspy install license <access code>`. The models in this workshop fit the
demo licence.
</details>

## 8. Checkpoint

- Which stage is the root, and why is that the default for a variable you do not annotate?
- How many linking rows does the storage create with 8 contiguous blocks, and how many with 8
  round-robin blocks?

## Next

[03 Structure detection](03-structure-detection.ipynb), which finds a structure in a model that
states none. Or [02 PyPSA](02-pypsa.ipynb) if you use PyPSA.

## Notes

- Linear, continuous models only. Nonlinear terms and discrete variables raise a `ValueError`
  naming how many were found.
- Equation types `=e=`, `=g=` and `=l=` map onto PIPS-IPM++ rows; anything else is refused.
- Maximisation is handled for you: objective and marginals are flipped back on the way out.